# TFT training on Colab — Drive-persistent, fast budget

**Setup:** Runtime → Change runtime type → T4 GPU.

All checkpoints (Optuna DB, model state, predictions) are saved to `/content/drive/MyDrive/dis_tft/` so a session disconnect doesn't lose progress. Re-run all cells after reconnect — anything already done is skipped.

Budget: 5 Optuna trials × 8 epochs each + 20-epoch final fit per direction. ~15–20 min total on T4.

In [ ]:
!pip install -q pytorch-forecasting lightning optuna pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ART_DIR = '/content/drive/MyDrive/dis_tft'
os.makedirs(ART_DIR, exist_ok=True)
print('persistent dir:', ART_DIR)

In [ ]:
!wget -q https://raw.githubusercontent.com/SaidkomilMS/dis-tft-data/main/dataset.parquet -O /content/dataset.parquet
!wget -q https://raw.githubusercontent.com/SaidkomilMS/dis-tft-data/main/dataset_meta.json -O /content/dataset_meta.json
!ls -la /content/dataset.parquet /content/dataset_meta.json

In [ ]:
import torch, json, numpy as np, pandas as pd
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
df = pd.read_parquet('/content/dataset.parquet')
META = json.loads(open('/content/dataset_meta.json').read())
df['hour_utc'] = pd.to_datetime(df['hour_utc'], utc=True)
df = df.sort_values('hour_utc').reset_index(drop=True)
df['time_idx'] = np.arange(len(df), dtype=np.int64)
df['group'] = 'usd_rub'
for c in ('turnover_usd', 'bank_rate', 'spread', 'spread_pct', 'ref_rate'):
    df[c] = df[c].astype('float64')
print('rows:', len(df), 'horizons:', META['horizons'], 'train_pool_end:', META['splits']['train_pool_end'])

In [ ]:
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

QUANTILES = [0.1, 0.5, 0.9]
MAX_ENC = 48
MAX_PRED = max(META['horizons'])
TRAIN_CUT = META['splits']['train_pool_end']
STORAGE = f'sqlite:///{ART_DIR}/optuna_tft.db'
N_TRIALS_TFT = 5

TIME_KNOWN = ['time_idx', 'hour_of_day', 'day_of_week', 'is_peak', 'is_weekend', 'ref_rate',
              'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'hour_of_week', 'sin_hour_of_week', 'cos_hour_of_week',
              'hours_from_peak', 'day_part', 'business_hour']
TIME_UNKNOWN = ['turnover_usd', 'bank_rate', 'spread', 'spread_pct']

def make_ds(target):
    return TimeSeriesDataSet(
        df[df.time_idx <= TRAIN_CUT],
        time_idx='time_idx', target=target, group_ids=['group'],
        max_encoder_length=MAX_ENC, max_prediction_length=MAX_PRED,
        static_categoricals=['group'],
        time_varying_known_reals=TIME_KNOWN,
        time_varying_unknown_reals=TIME_UNKNOWN,
        target_normalizer=GroupNormalizer(groups=['group'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=True,
    )

In [ ]:
def search_hp(target, name, n_trials=N_TRIALS_TFT, max_epochs=8):
    cache = f'{ART_DIR}/best_{name}.json'
    if os.path.exists(cache):
        print(f'✓ {name}: best params already saved')
        return json.loads(open(cache).read())
    train_ds = make_ds(target)
    val_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=False, stop_randomization=True)
    train_dl = train_ds.to_dataloader(train=True, batch_size=128, num_workers=2)
    val_dl = val_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    def objective(trial):
        params = {
            'hidden_size': trial.suggest_int('hidden_size', 16, 64, step=16),
            'attention_head_size': trial.suggest_int('attention_head_size', 1, 4),
            'dropout': trial.suggest_float('dropout', 0.05, 0.3),
            'hidden_continuous_size': trial.suggest_int('hidden_continuous_size', 8, 24, step=8),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3, log=True),
        }
        pl.seed_everything(7)
        model = TemporalFusionTransformer.from_dataset(
            train_ds, **params, loss=QuantileLoss(quantiles=QUANTILES), log_interval=0)
        es = EarlyStopping(monitor='val_loss', patience=2, mode='min')
        trainer = pl.Trainer(max_epochs=max_epochs, accelerator='auto', gradient_clip_val=0.5,
                              callbacks=[es], enable_model_summary=False, enable_progress_bar=False, logger=False)
        trainer.fit(model, train_dl, val_dl)
        return float(trainer.callback_metrics.get('val_loss', float('inf')))
    study = optuna.create_study(study_name=f'tft_{name}', storage=STORAGE, load_if_exists=True,
                                  direction='minimize', sampler=optuna.samplers.TPESampler(seed=7))
    completed = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    remaining = max(0, n_trials - completed)
    print(f'optuna {name!r}: {completed}/{n_trials} done, running {remaining} more')
    if remaining > 0:
        study.optimize(objective, n_trials=remaining,
                       callbacks=[lambda s, t: print(f'  trial {t.number+1} val={t.value:.4f}')])
    best = study.best_params
    open(cache, 'w').write(json.dumps(best, indent=2))
    print(f'best {name}: {best}')
    return best

def train_final(target, name, params, max_epochs=20):
    pred_npy = f'{ART_DIR}/preds_{name}.npy'
    idx_npy  = f'{ART_DIR}/decoder_idx_{name}.npy'
    if os.path.exists(pred_npy) and os.path.exists(idx_npy):
        print(f'✓ {name}: predictions already saved')
        return np.load(pred_npy), np.load(idx_npy)
    train_ds = make_ds(target)
    val_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=False, stop_randomization=True)
    train_dl = train_ds.to_dataloader(train=True, batch_size=128, num_workers=2)
    val_dl = val_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    pl.seed_everything(7)
    model = TemporalFusionTransformer.from_dataset(
        train_ds, **params, loss=QuantileLoss(quantiles=QUANTILES), log_interval=0)
    es = EarlyStopping(monitor='val_loss', patience=4, mode='min')
    ckpt = ModelCheckpoint(monitor='val_loss', mode='min', dirpath=ART_DIR, filename=f'tft_{name}')
    trainer = pl.Trainer(max_epochs=max_epochs, accelerator='auto', gradient_clip_val=0.5,
                          callbacks=[es, ckpt], enable_model_summary=False, logger=False)
    trainer.fit(model, train_dl, val_dl)
    full_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=True, stop_randomization=True)
    full_dl = full_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    preds = trainer.predict(model, full_dl, return_predictions=True)
    out = torch.cat(preds, dim=0).cpu().numpy()
    decoder_idx = full_ds.x_to_index(full_ds)['time_idx'].values
    np.save(pred_npy, out)
    np.save(idx_npy, decoder_idx)
    return out, decoder_idx

In [ ]:
best_A = search_hp('turnover_usd', 'A_turnover')
out_A, decoder_idx_A = train_final('turnover_usd', 'A_turnover', best_A)
print('A shape:', out_A.shape)

In [ ]:
best_B = search_hp('bank_rate', 'B_rate')
out_B, decoder_idx_B = train_final('bank_rate', 'B_rate', best_B)
print('B shape:', out_B.shape)

In [ ]:
records = []
for h in META['horizons']:
    cum = out_A[:, :h, :].sum(axis=1)
    records.append(pd.DataFrame({
        'row_idx': decoder_idx_A,
        'p10': cum[:,0], 'p50': cum[:,1], 'p90': cum[:,2],
        'direction': 'A', 'horizon': h,
    }))
for h in META['horizons']:
    records.append(pd.DataFrame({
        'row_idx': decoder_idx_B,
        'p10': out_B[:,0,0], 'p50': out_B[:,0,1], 'p90': out_B[:,0,2],
        'direction': 'B', 'horizon': h,
    }))
final = pd.concat(records, ignore_index=True)
out_path = f'{ART_DIR}/predictions_tft.parquet'
final.to_parquet(out_path, index=False)
print('SAVED to Drive:', out_path, 'rows:', len(final))
print('Open https://drive.google.com/drive/my-drive and look for the dis_tft folder.')